# 08 — Transfer analysis

Run this after the formal validation, held-out test, and preferably LOIO notebooks.
It summarizes transfer support/source relations for the large-felid formal package.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)
print("PROJECT_ROOT:", PROJECT_ROOT, flush=True)

In [ ]:
from carnivore_reconstruction.timing import TimerLog, status
from carnivore_reconstruction.formal_large_felid import read_table_auto, summarize_with_linear, prepare_formal_task_tables
from carnivore_reconstruction.transfer import transfer_scenario_counts, summarize_metrics_by_transfer

MODEL_DIR = PROJECT_ROOT / "outputs" / "pretrained_model"
VALIDATION_DIR = PROJECT_ROOT / "outputs" / "formal_validation"
TEST_DIR = PROJECT_ROOT / "outputs" / "formal_heldout_test"
LOIO_DIR = PROJECT_ROOT / "outputs" / "leave_one_individual_out"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "transfer_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timer = TimerLog()

In [ ]:
status("Transfer 1/4: loading task and metric tables")
task_table = read_table_auto(MODEL_DIR / "task_table.parquet")
formal_split = pd.read_csv(MODEL_DIR / "formal_individual_split_table.csv")
task_table, _, formal_split = prepare_formal_task_tables(
    task_table,
    None,
    formal_split,
    seed=20260625,
    label="formal_task_table",
)
validation_metrics = pd.read_csv(VALIDATION_DIR / "formal_validation_task_metrics.csv") if (VALIDATION_DIR / "formal_validation_task_metrics.csv").exists() else pd.DataFrame()
test_metrics = pd.read_csv(TEST_DIR / "formal_heldout_test_task_metrics.csv") if (TEST_DIR / "formal_heldout_test_task_metrics.csv").exists() else pd.DataFrame()
loio_metrics = pd.read_csv(LOIO_DIR / "loio_task_metrics.csv") if (LOIO_DIR / "loio_task_metrics.csv").exists() else pd.DataFrame()

print("task_table:", task_table.shape)
print("validation_metrics:", validation_metrics.shape)
print("test_metrics:", test_metrics.shape)
print("loio_metrics:", loio_metrics.shape)

In [ ]:
status("Transfer 2/4: saving transfer scenario counts")
train_to_validation = transfer_scenario_counts(task_table, source_split="train", target_split="validation", exclude_same_animal=True)
train_to_test = transfer_scenario_counts(task_table, source_split="train", target_split="test", exclude_same_animal=True)

train_to_validation.to_csv(OUTPUT_DIR / "transfer_scenario_counts_train_to_validation.csv", index=False)
train_to_test.to_csv(OUTPUT_DIR / "transfer_scenario_counts_train_to_test.csv", index=False)
formal_split.to_csv(OUTPUT_DIR / "formal_individual_split_table.csv", index=False)

display(train_to_test.head(30))

In [ ]:
status("Transfer 3/4: metric summaries by transfer relation and source relation")
for name, metrics in [("validation", validation_metrics), ("heldout_test", test_metrics), ("loio", loio_metrics)]:
    if metrics is None or metrics.empty:
        continue

    if "best_train_transfer_relation" in metrics.columns:
        summary = summarize_with_linear(metrics, ["best_train_transfer_relation", "method"])
        summary.to_csv(OUTPUT_DIR / f"{name}_method_summary_by_best_train_transfer_relation.csv", index=False)

    source_cols = [c for c in metrics.columns if "source" in c.lower() and "relation" in c.lower()]
    if source_cols:
        for c in source_cols:
            try:
                src_summary = summarize_with_linear(metrics, [c, "method"])
                src_summary.to_csv(OUTPUT_DIR / f"{name}_method_summary_by_{c}.csv", index=False)
            except Exception as exc:
                print(f"Could not summarize {name} by {c}: {exc}")

    # Species/setting summary is useful even when strict transfer labels are sparse.
    species_cols = [c for c in ["taxon", "setting_name", "method"] if c in metrics.columns]
    if species_cols:
        summarize_with_linear(metrics, species_cols).to_csv(OUTPUT_DIR / f"{name}_species_setting_method_summary.csv", index=False)

print("Saved transfer summaries to:", OUTPUT_DIR)

In [ ]:
status("Transfer 4/4: concise transfer report")
report_lines = [
    "Large-felid formal transfer analysis",
    "====================================",
    "",
    "Formal split:",
    formal_split.groupby(["taxon", "split"], dropna=False).size().reset_index(name="n_individuals").to_string(index=False),
    "",
    "Train → test transfer scenario counts:",
    train_to_test.to_string(index=False),
]

if (OUTPUT_DIR / "heldout_test_method_summary_by_best_train_transfer_relation.csv").exists():
    report_lines += [
        "",
        "Held-out test summary by best train transfer relation:",
        pd.read_csv(OUTPUT_DIR / "heldout_test_method_summary_by_best_train_transfer_relation.csv").to_string(index=False),
    ]

(OUTPUT_DIR / "transfer_analysis_report.txt").write_text("\n".join(report_lines), encoding="utf-8")
print("\n".join(report_lines[:8]))